In [5]:
import numpy as np
from Dynamo.star import Star
from Dynamo.create_data import generate_simdata, simulate_one
import logging

In [ ]:
from Dynamo.phoenix import download_phoenix_models

download_phoenix_models(
    base_dir='/Users/philvanlane/Documents/Dynamo/.dynamo/models/Phoenix_mu',
    metallicity_range=(-0.5, 0.5),   # solar-like metallicity
    temperature_range=(2500, 7000),  # FGK dwarfs; expand if needed
    logg_range=(4.0, 5.5),           # main sequence
    max_workers=4,
)

Using 4 parallel workers
Checking available metallicities from server...
Available metallicities on server: [-4.0, -3.0, -2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0]
Temperature range: (2500, 7000) K
Log g range: (4.0, 5.5)

Gathering files for metallicity Z-0.5...
Found 179 files matching criteria

Gathering files for metallicity Z-0.0...
Found 179 files matching criteria

Gathering files for metallicity Z+0.5...
Found 177 files matching criteria

Total files to download: 535


In [8]:


# 1. Create a Star object (reads test_star.conf for defaults)
sm = Star(conf_file_path='/Users/philvanlane/Documents/lc_ae/ilay_sims/test_star.conf')

# 2. Set stellar parameters manually (overrides conf values)
sim_row = {
    'mass': 1.0,          # Msun
    'age': 2.0,           # Gyr
    'FeH': 0.0,           # metallicity
    'alpha/H': 0.0,
    'Teff': 5778.0,       # K
    'logg': 4.44,
    'L': 1.0,             # Lsun
    'R': 1.0,             # Rsun
    'Period': 25.0,       # rotation period (days)
    'Inclination': np.pi/2,  # radians (pi/2 = equator-on)
    'Shear': 0.2,         # differential rotation coefficient
    'Activity Rate': 1.0,
    'Cycle Length': 11.0, # years
    'Cycle Overlap': 1.0,
    'Spot Min': 5.0,      # min spot latitude (deg)
    'Spot Max': 40.0,     # max spot latitude (deg)
    'Decay Time': 5.0,    # spot lifetime in units of Prot
    'Butterfly': True,
    'convective_shift': 1.0,
    'Distance': 100.0,    # pc
    'CDPP': 50.0,         # white noise (ppm); 0 = no noise
    'Outlier Rate': 0.0,
    'Flicker Time Scale': 0.0,
    # Planet (set simulate_planet=0 to skip)
    'simulate_planet': 0
}

sm.set_stellar_parameters(sim_row)
sm.set_planet_parameters(sim_row)

# 3. Generate spot map
ndays = 27  # one TESS sector
sm.generate_spot_map(ndays=ndays)

# 4. Build time array — TESS 2-min cadence = 0.001389 days
cadence = 0.001389  # days  (use 0.020833 for 30-min)
t = np.linspace(0, ndays, int(ndays / cadence))
t_dict = {'TESS': t}

# 5. Run simulation
sm.compute_forward(t=t_dict)

# 6. Extract light curve
time_out, flux_out = sm.results['lcs']['TESS']


number of spots:  20 activity:  1.0
number of grid rings:  29

computing forward with: 20 spots and 0 planets


SystemExit: Error: No PHOENIX SpecInt files found in /Users/philvanlane/Documents/Dynamo/.dynamo/models/Phoenix_mu